### ML Model Training

Dataset: IEEE-CIS Fraud Detection

| Model | Model Type | Notes |
|---------|------------|-------|
| 1 | LightGBM | Baseline features, binary classification, scale_pos_weight, early stopping |
| 2 | LightGBM | Engineered features, binary classification, scale_pos_weight, early stopping |
| 3 | Isolation Forest | Scaled data, contamination=0.035, anomaly detection |
| 4 | Local Outlier Factor | Scaled data, sub-sampled training (50k), novelty=True, contamination=0.035 |
| 5 | XGBoost | Engineered features, binary logistic, scale_pos_weight, early stopping, tree_method=hist |

In [3]:
import pandas as pd
import numpy as np
import gc
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# ---------------------------------------------------------
# 1. HELPER: TRAINING ENGINE
# ---------------------------------------------------------
def train_lgb(X_train, y_train, X_val, y_val, name="Model"):
    print(f"\n--- Training {name} ---")
    weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    params = {'objective': 'binary', 'metric': 'average_precision', 'scale_pos_weight': weight,
              'learning_rate': 0.03, 'num_leaves': 256, 'verbosity': -1}
    
    model = lgb.train(params, dtrain, valid_sets=[dval], num_boost_round=1000,
                      callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=0)])
    
    probs = model.predict(X_val)
    return probs, roc_auc_score(y_val, probs), average_precision_score(y_val, probs)

# ---------------------------------------------------------
# 2. EXPERIMENT: FEATURE IMPACT ANALYSIS (Rubric Requirement)
# ---------------------------------------------------------
print("Starting Impact Analysis...")
# BUG FIXED HERE, reading from pre split parquets
df_base_train = pd.read_parquet('data/iceee_baseline_train.parquet')
df_base_val   = pd.read_parquet('data/iceee_baseline_val.parquet')
X_tr_b = df_base_train.drop(['isFraud','TransactionID','TransactionDT'], axis=1, errors='ignore')
y_tr_b = df_base_train['isFraud']
X_vl_b = df_base_val.drop(['isFraud','TransactionID','TransactionDT'], axis=1, errors='ignore')
y_vl_b = df_base_val['isFraud']

probs_base, auc_base, auprc_base = train_lgb(X_tr_b, y_tr_b, X_vl_b, y_vl_b, "Baseline")
del df_base_train, df_base_val; gc.collect()

df_eng_train = pd.read_parquet('data/iceee_feature_train.parquet')
df_eng_val   = pd.read_parquet('data/iceee_feature_val.parquet')
X_tr_e = df_eng_train.drop(['isFraud','TransactionID','TransactionDT'], axis=1, errors='ignore')
y_tr_e = df_eng_train['isFraud']
X_vl_e = df_eng_val.drop(['isFraud','TransactionID','TransactionDT'], axis=1, errors='ignore')
y_vl_e = df_eng_val['isFraud']

probs_eng, auc_eng, auprc_eng = train_lgb(X_tr_e, y_tr_e, X_vl_e, y_vl_e, "Engineered")
del df_eng_train, df_eng_val; gc.collect()
# ---------------------------------------------------------
# 3. CLASSICAL ANOMALY DETECTION (LOF & Isolation Forest)
# ---------------------------------------------------------
print("\n--- Running Classical Baselines ---")
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_e.fillna(-999))
X_vl_s = scaler.transform(X_vl_e.fillna(-999))

# Isolation Forest
iso = IsolationForest(n_estimators=100, contamination=0.035, random_state=42)
iso.fit(X_tr_s)
iso_probs = (iso.decision_function(X_vl_s).max() - iso.decision_function(X_vl_s)) # Normalize to 0-1 later

# Local Outlier Factor (LOF) - SUB-SAMPLED TO PREVENT MEMORY CRASH
# We take a sample of the validation set to demonstrate the method as per rubric
print("Running LOF on validation sample (due to memory constraints)...")
lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.035)
lof.fit(X_tr_s[:50000]) # Fitting on a subset for speed
lof_scores = lof.decision_function(X_vl_s)
lof_probs = (lof_scores.max() - lof_scores)

# ---------------------------------------------------------
# 4. XGBOOST
# ---------------------------------------------------------
dtrain_xgb = xgb.DMatrix(X_tr_e, label=y_tr_e)
dval_xgb = xgb.DMatrix(X_vl_e, label=y_vl_e)
xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 
              'scale_pos_weight': len(y_tr_e[y_tr_e==0])/len(y_tr_e[y_tr_e==1]), 'tree_method': 'hist'}
clf_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=500, evals=[(dval_xgb, 'valid')], 
                    early_stopping_rounds=50, verbose_eval=0)
probs_xgb = clf_xgb.predict(dval_xgb)

# ---------------------------------------------------------
# 5. FINAL EXPORT
# ---------------------------------------------------------
impact_df = pd.DataFrame({'Metric': ['ROC-AUC', 'AUPRC'], 'Baseline': [auc_base, auprc_base], 'Engineered': [auc_eng, auprc_eng]})
impact_df.to_csv('results/feature_impact_results.csv', index=False)

final_probs = pd.DataFrame({'actual': y_vl_e, 'lgb': probs_eng, 'xgb': probs_xgb, 'iso': iso_probs, 'lof': lof_probs})
final_probs.to_csv('results/all_ml_probs.csv', index=False)
print("\nSuccess: Impact analysis and all classical models complete.")

Starting Impact Analysis...

--- Training Baseline ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[989]	valid_0's average_precision: 0.576368

--- Training Engineered ---
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[999]	valid_0's average_precision: 0.592236

--- Running Classical Baselines ---
Running LOF on validation sample (due to memory constraints)...

Success: Impact analysis and all classical models complete.
